# Introduction to MongoDB and PyMongo
*This notebook introduces the MongoDB database and the methods used to interact with it from Python through the PyMongo client library.*

---

## What is MongoDB?

**MongoDB** is a **document-oriented NoSQL database** designed to store and manage data that does not fit well into rigid, relational table structures.

Instead of rows and columns, MongoDB stores data as **documents**, using a JSON-like format called **BSON** (Binary JSON).

Each document is a self-contained data structure made up of:
- key–value pairs  
- nested objects  
- arrays  
- optional or missing fields  

Documents are grouped into **collections**, which are roughly analogous to tables in relational databases.

MongoDB was designed for situations where:
- data structure changes over time  
- not all records look the same  
- nesting is natural  
- schema flexibility is more important than strict constraints  

---

## BSON: How MongoDB Stores Data

MongoDB stores data internally using a format called **BSON** (Binary JSON).

BSON is not a different data model from JSON — it is a **binary representation** of JSON-like documents that is optimized for storage efficiency and performance.

From the user’s perspective:
- documents look like JSON  
- but are stored and transmitted as BSON  

### Why MongoDB uses BSON

JSON is widely understood and allows for flexible schemas. However, it does not enforce explicit data types, and parsing text-based JSON can be relatively slow.

BSON extends JSON by adding:
- explicit data types  
- native support for dates and timestamps  
- support for large integers and decimals  

This allows MongoDB to store and query data **efficiently and unambiguously**, while preserving the flexibility of a document-based model.

---

## What is PyMongo?

**Pymongo** is the official **Python driver for MongoDB**.
MongoDB is a database system that runs as a separate service.
To interact with it from Python, a client library is required.

PyMongo allows applications to:
- connect to a MongoDB server
- create and manage databases and collections
- insert, query, update, and delete documents
- work with MongoDB data using native Python data structures
- automatically convert between BSON and Python dictionaries

PyMongo acts as the bridge between Python and MongoDB.

PyMongo does not replace tools like Spark, which are designed for distributed computation. It primary role is to provide a direct interaction with the database.

---

## Implementation in code

### Creating a client
To interact with MongoDB from Python, we first need to create a client.

The client represents a connection to a MongoDB server and is the entry point for all database operations.

In [1]:
from pymongo import MongoClient

myclient = MongoClient("mongodb://localhost:27017")

At this point:
- no data has been read
- no query has been executed
- no database has been selected yet

The client simply establishes a connection configuration. You usually need only one client per application.

#### Connection URI:
`mongodb://localhost:27017`

- `mongodb://` is the MongoDB protocol (the URI always starts with this)
- `localhost` is the server address; for a remote connection, this would be replaced by the host name or IP address of the MongoDB server
- `27017` is the default MongoDB port; for a remote connection, this value may differ depending on the server configuration

---

### Databases, Collections, and Documents
In MongoDB, data is organized into three levels:

```
MongoDB Server
 └── Database
      └── Collection
           └── Document
```

- A *database* is a "logical container" that groups related data.
- A *collection* is a "group of documents" within a database, roughly analogous to a table. it does not enforce a fixed schema by default.
- A *document* is a single record stored as a BSON object (JSON-like structure). It is represented as a Python dictionary when using PyMongo and always contains an `_id` field that uniquely identifies it.


#### Accessing a database

In [2]:
mydatabase = myclient.testdb

- `testdb` does not need to exist beforehand
- MongoDB creates databases lazily
- **databases** are created only when data is inserted

#### Accesing a collection

In [3]:
mycollection = mydatabase.people

- `people` does not need to exist yet
- **collections** are created when the first document is inserted

---

### CRUD Operations
*(Create, Read, Update & Delete)*

### Inserting Data into a collection

In [4]:
document = {
    "name": "Alice",
    "email": "alice@example.com",
    "city": "Oslo"
}

mycollection.insert_one(document)

InsertOneResult(ObjectId('698b302ad0ed5209f80600dd'), acknowledged=True)

`insert_one()` inserts exactly one document.
It returns an object containing the inserted document’s `_id`. If an `_id` is not provided, MongoDB generates one automatically

At this moment:
- the database `testdb` is created
- the collection `people` is created
- the document is stored in MongoDB

This is the first time anything actually exists. At this point we can verify if there is any data in our collection:

In [5]:
mycollection.find_one()

# `find_one()` returns one document from the collection as a Python dictionary,
#  including the automatically generated _id.

{'_id': ObjectId('698b2b160cd5e8e326b67d8f'),
 'name': 'Alice',
 'email': 'alice@example.com',
 'city': 'Oslo'}

We can also insert several documents at once and specify our own `_id` field:

In [8]:
peopleList=[
            {"_id":1, "name":"James", "email":"james@example.com"},
            {"_id":2, "name":"Joger", "email":"joger@example.com"},
            {"_id":3, "name":"Emma", "email":"emma@example.com"},
            {"_id":4, "name":"Mary", "email":"mary@example.com"},
            {"_id":5, "name":"Olav", "email":"Olav@example.com"},
            {"_id":6, "name":"Embrik", "email":"Embrik@example.com"}
            ]

mycollection.insert_many(peopleList)

InsertManyResult([1, 2, 3, 4, 5, 6], acknowledged=True)

In [10]:
for i in mycollection.find():
    print(i)

{'_id': ObjectId('698b2b160cd5e8e326b67d8f'), 'name': 'Alice', 'email': 'alice@example.com', 'city': 'Oslo'}
{'_id': ObjectId('698b302ad0ed5209f80600dd'), 'name': 'Alice', 'email': 'alice@example.com', 'city': 'Oslo'}
{'_id': 1, 'name': 'James', 'email': 'james@example.com'}
{'_id': 2, 'name': 'Joger', 'email': 'joger@example.com'}
{'_id': 3, 'name': 'Emma', 'email': 'emma@example.com'}
{'_id': 4, 'name': 'Mary', 'email': 'mary@example.com'}
{'_id': 5, 'name': 'Olav', 'email': 'Olav@example.com'}
{'_id': 6, 'name': 'Embrik', 'email': 'Embrik@example.com'}


`find()` returns a *cursor* which is an iterable object that lazily retrieves documents from the database. Documents are fetched in batches and returned one at a time when iterating over the cursor. This avoids loading all data into memory at once.